# core

> Server startup: bring the boopiter web app up on a port (or cleanly relaunch it), refusing to stomp on a port held by something that isn't boopiter. Precompiles Tailwind once, then starts the server; `launch` is the entry point the CLI and notebook both call.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, subprocess, sys, time, urllib.request, urllib.error, secrets
from pathlib import Path
from typing import Annotated
from fastcore.script import call_parse

## Port management

Before starting, boopiter checks who owns the target port: `_port_owner` pings it to tell a running boopiter from someone else from nobody, and `_kill_port` frees a port held by a *previous boopiter* run -- so relaunching is safe but an unrelated service is never killed.

In [ ]:
#| export
def _port_owner(port:int):
    "Return 'boopiter' if boopiter answers on `port`, 'other' if something else does, None if nothing does."
    try:
        with urllib.request.urlopen(f'http://localhost:{port}/_boopiter_ping', timeout=1.0) as r:
            text = r.read().decode().strip()
        return 'boopiter' if text == 'boopiter' else 'other'
    except urllib.error.HTTPError:
        return 'other'   # got a real HTTP response, just not our ping -- something else is there
    except (ConnectionRefusedError, urllib.error.URLError):
        return None      # nothing is accepting connections there
    except Exception:
        return 'other'   # some other failure mode -- be conservative, don't touch it

In [ ]:
#| export
def _kill_port(port:int):
    try:
        pids = subprocess.run(['lsof', '-ti', f'tcp:{port}'], capture_output=True, text=True).stdout.split()
    except FileNotFoundError:
        pids = []
    for pid in pids: subprocess.run(['kill', '-9', pid])
    if pids: time.sleep(0.3)

## Tailwind precompile

`_build_tailwind` compiles a static Tailwind CSS file once at launch, replacing the CDN's in-browser JIT compiler -- faster page loads and no runtime CDN dependency.

In [ ]:
#| export
def _build_tailwind():
    "Precompile a static Tailwind CSS file once per launch, replacing the CDN's in-browser JIT compiler (which recompiles on every htmx DOM update -- a major source of per-interaction lag)."
    pkg_dir = Path(__file__).parent
    inp, out = pkg_dir/'static/tw_input.css', pkg_dir/'static/tailwind.css'
    venv_bin = Path(sys.executable).parent/'tailwindcss'  # pytailwindcss installs its binary alongside python, not necessarily on PATH
    exe = str(venv_bin) if venv_bin.exists() else 'tailwindcss'
    try:
        subprocess.run([exe, '-i', str(inp), '-o', str(out), '--minify', '--cwd', str(pkg_dir)],
                       capture_output=True, timeout=60, check=True)
    except Exception as e:
        print(f"Tailwind precompile failed ({e}); falling back to the slower CDN JIT build.", file=sys.stderr)

## Launching

`launch` ties it together -- free the port if it's ours, build the CSS, start the server. It's the `@call_parse` entry point exposed on the command line.

In [ ]:
#| export
@call_parse
def launch(
    nbfile: Annotated[str, {'opt': False, 'nargs': '?'}] = None,  # .ipynb file to load on startup
    port: int = 8000,  # the port to serve boopiter on
    no_auth: bool = False,  # skip token-gating the server -- ONLY on a network you fully trust; every code cell already runs with this process's own permissions, so an unauthed boopiter is equivalent to handing out a shell
):
    "Launch (or relaunch) the boopiter server; refuses to touch a port held by something that isn't boopiter"
    owner = _port_owner(port)
    if owner == 'other':
        print(f"Port {port} is already in use by something that isn't boopiter -- aborting.", file=sys.stderr)
        sys.exit(1)
    if owner == 'boopiter':
        print(f"Killing previous boopiter instance on port {port}...")
        _kill_port(port)
    _build_tailwind()
    if nbfile:
        from . import cells as _cells
        try:
            _cells.load_notebook(nbfile)
        except FileNotFoundError:
            print(f"No such notebook: {nbfile} -- starting with a blank notebook instead.", file=sys.stderr)
    os.environ['BOOPITER_PORT'] = str(port)  # lets a self-restart (see cells.restart_server) rebind the same port
    # BOOPITER_TOKEN/BOOPITER_NO_AUTH are read straight from the environment on every launch (including
    # cells.restart_server's os.execv self-restart, which inherits the environment unchanged) -- so a
    # token generated here survives restarts instead of silently invalidating every open browser session.
    if no_auth: os.environ['BOOPITER_NO_AUTH'] = '1'
    if os.environ.get('BOOPITER_NO_AUTH') == '1':
        print(f"\n⚠️  boopiter is running WITHOUT auth (--no_auth) -- anyone who can reach port {port} "
              f"has full code execution as you. Only do this on a network you fully trust.\n", file=sys.stderr)
    else:
        token = os.environ.get('BOOPITER_TOKEN') or secrets.token_urlsafe(24)
        os.environ['BOOPITER_TOKEN'] = token
        print(f"\nboopiter is token-gated. Open:\n\n    http://localhost:{port}/?token={token}\n")
    import uvicorn
    uvicorn.run('boopiter.cells:app', host='0.0.0.0', port=port)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()